# Histogramas e tabelas comparativas do escalonamento (Anexo A.1)

Este notebook gera as figuras da seção **A.1.5** e as duas tabelas da seção **A.1.6** do documento.
Ele é o par do `escalonamento_anexo_a1.ipynb`, que trata das seções A.1.2 a A.1.4: as equações e as
constantes vêm de lá e são apenas reaplicadas aqui, para que as figuras e as tabelas saiam dos mesmos
números publicados.

**Base de entrada.** Lê `data/processed/base_analitica.parquet`, gerado por
`notebooks/pre-processamento.ipynb`. A pasta `data/` não é versionada, por compromisso entre o Inteli
e o parceiro, então o arquivo precisa existir localmente antes da execução. Se ele não for encontrado,
a base é reconstruída a partir de `data/raw` com `scripts/preprocessamento_nps.py`. A contagem de
484.915 registros é verificada por `assert`.

**Duas garantias contra número velho.** As constantes reaplicadas na seção 1 são conferidas contra a
base carregada, e os dez registros das tabelas da seção 3 são selecionados por `RESPONDENT_ID`, nunca
por posição. O motivo está na seção 3.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = next(
    (c for c in [Path('.'), Path('..'), Path('/content')] if (c / 'data').exists() or (c / 'scripts').exists()),
    Path('.'),
)

caminho_base = RAIZ / 'data' / 'processed' / 'base_analitica.parquet'

if caminho_base.exists():
    df = pd.read_parquet(caminho_base)
else:
    sys.path.insert(0, str((RAIZ / 'scripts').resolve()))
    from preprocessamento_nps import integrar_bases, preparar_base_analitica

    df_bruto, _ = integrar_bases(RAIZ / 'data' / 'raw')
    df = preparar_base_analitica(df_bruto)

assert len(df) == 484915, f'a base analítica deveria ter 484.915 registros e tem {len(df)}'

PASTA_ASSETS = RAIZ / 'assets'
PASTA_ASSETS.mkdir(exist_ok=True)

print('Registros na base analítica:', f'{len(df):,}'.replace(',', '.'))

## 1. Reaplicação das equações da seção A.1.4

As mesmas constantes publicadas no documento, com as quatro casas decimais. A conferência de que elas
reproduzem a coluna escalonada está no `escalonamento_anexo_a1.ipynb`, na seção intitulada
*Conferência manual entre a equação e a coluna gerada*, e não é repetida aqui. A referência é pelo
título e não pelo número da seção porque aquele notebook é renumerado sempre que uma seção nova é
inserida antes dela.

As constantes são reaplicadas, e não recalculadas, para que estas figuras saiam dos mesmos números
publicados na seção A.1.3. Por isso elas são conferidas contra a base carregada logo abaixo: se a base
mudar, a execução para em vez de gerar figura com constante velha.

In [ ]:
MEDIA_TEMPO_VOO, DESVIO_TEMPO_VOO = 207.1565, 208.2920
MEDIA_ATRASO_CHEGADA, DESVIO_ATRASO_CHEGADA = 25.6551, 136.4553
MINIMO_QTDE_VIAGENS_12M, MAXIMO_QTDE_VIAGENS_12M = 0, 107

# Tolerancia de meia unidade na quarta casa, que e a precisao com que as constantes foram publicadas.
TOLERANCIA = 5e-5
conferencias = [
    ('media de TEMPO_VOO', df['TEMPO_VOO'].mean(), MEDIA_TEMPO_VOO),
    ('desvio de TEMPO_VOO', df['TEMPO_VOO'].std(ddof=0), DESVIO_TEMPO_VOO),
    ('media de ATRASO_CHEGADA', df['ATRASO_CHEGADA'].mean(), MEDIA_ATRASO_CHEGADA),
    ('desvio de ATRASO_CHEGADA', df['ATRASO_CHEGADA'].std(ddof=0), DESVIO_ATRASO_CHEGADA),
    ('minimo de QTDE_VIAGENS_12M', df['QTDE_VIAGENS_12M'].min(), MINIMO_QTDE_VIAGENS_12M),
    ('maximo de QTDE_VIAGENS_12M', df['QTDE_VIAGENS_12M'].max(), MAXIMO_QTDE_VIAGENS_12M),
]
for nome, calculado, publicado in conferencias:
    assert abs(float(calculado) - publicado) < TOLERANCIA, (
        f'{nome}: a base dá {float(calculado):.4f} e o documento publica {publicado}'
    )
print(f'{len(conferencias)} constantes da seção A.1.3 conferidas contra a base')

df['TEMPO_VOO_ESC'] = (df['TEMPO_VOO'] - MEDIA_TEMPO_VOO) / DESVIO_TEMPO_VOO
df['ATRASO_CHEGADA_ESC'] = (df['ATRASO_CHEGADA'] - MEDIA_ATRASO_CHEGADA) / DESVIO_ATRASO_CHEGADA
df['QTDE_VIAGENS_12M_ESC'] = (
    (df['QTDE_VIAGENS_12M'] - MINIMO_QTDE_VIAGENS_12M)
    / (MAXIMO_QTDE_VIAGENS_12M - MINIMO_QTDE_VIAGENS_12M)
)

df[['TEMPO_VOO_ESC', 'ATRASO_CHEGADA_ESC', 'QTDE_VIAGENS_12M_ESC']].describe().round(4)

## 2. Histogramas antes e depois (figuras A.1 a A.8 da seção A.1.5)

Os seis primeiros gráficos usam a mesma quantidade de classes e o mesmo estilo, porque o ponto da
comparação é que só o eixo horizontal muda. Cada arquivo é gravado em `assets/` com o nome citado no
documento.

As figuras A.7 e A.8 repetem o par de `ATRASO_CHEGADA` com o eixo vertical em escala logarítmica. Com
60 classes sobre um intervalo de 0 a 4.319 minutos, a primeira classe tem cerca de 72 minutos de
largura e concentra quase toda a base, o que achata visualmente o resto da distribuição na escala
linear. A escala logarítmica torna a cauda visível sem alterar o argumento da silhueta idêntica, que
continua valendo porque os dois lados do par usam a mesma escala.

In [ ]:
COR = '#1f5c99'
CLASSES = 60

FIGURAS = [
    ('hist_tempo_voo_original', 'TEMPO_VOO', 'TEMPO_VOO antes do escalonamento', 'Duração do deslocamento (minutos)', False),
    ('hist_tempo_voo_padronizado', 'TEMPO_VOO_ESC', 'TEMPO_VOO após padronização', 'Escore z', False),
    ('hist_atraso_chegada_original', 'ATRASO_CHEGADA', 'ATRASO_CHEGADA antes do escalonamento', 'Atraso na chegada (minutos)', False),
    ('hist_atraso_chegada_padronizado', 'ATRASO_CHEGADA_ESC', 'ATRASO_CHEGADA após padronização', 'Escore z', False),
    ('hist_qtde_viagens_original', 'QTDE_VIAGENS_12M', 'QTDE_VIAGENS_12M antes do escalonamento', 'Viagens em 12 meses', False),
    ('hist_qtde_viagens_normalizado', 'QTDE_VIAGENS_12M_ESC', 'QTDE_VIAGENS_12M após normalização', 'Valor normalizado (0 a 1)', False),
    ('hist_atraso_chegada_original_log', 'ATRASO_CHEGADA', 'ATRASO_CHEGADA antes do escalonamento, eixo y logarítmico', 'Atraso na chegada (minutos)', True),
    ('hist_atraso_chegada_padronizado_log', 'ATRASO_CHEGADA_ESC', 'ATRASO_CHEGADA após padronização, eixo y logarítmico', 'Escore z', True),
]

for arquivo, coluna, titulo, rotulo_x, logaritmico in FIGURAS:
    figura, eixo = plt.subplots(figsize=(9, 5.4), dpi=100)
    eixo.hist(df[coluna].dropna(), bins=CLASSES, color=COR, edgecolor='white', linewidth=0.5)
    if logaritmico:
        eixo.set_yscale('log')
        eixo.set_ylabel('Frequência (escala logarítmica)')
    else:
        eixo.set_ylabel('Frequência')
    eixo.set_title(titulo, fontsize=13, pad=14)
    eixo.set_xlabel(rotulo_x)
    eixo.grid(axis='y', color='#dddddd', linewidth=0.8)
    eixo.set_axisbelow(True)
    for lado in ('top', 'right'):
        eixo.spines[lado].set_visible(False)
    figura.tight_layout()
    figura.savefig(PASTA_ASSETS / f'{arquivo}.png')
    plt.close(figura)

print(f'{len(FIGURAS)} figuras gravadas em {PASTA_ASSETS}')

# Origem do numero citado na secao A.1.5 sobre a classe que contem os voos pontuais.
contagens, bordas = np.histogram(df['ATRASO_CHEGADA'].dropna(), bins=CLASSES)
largura = bordas[1] - bordas[0]
zeros = int((df['ATRASO_CHEGADA'] == 0).sum())
print(
    f'ATRASO_CHEGADA: largura da classe = {largura:.2f} min; '
    f'primeira classe [0; {bordas[1]:.2f}) = {contagens[0]} registros '
    f'({contagens[0] / len(df) * 100:.2f}% da base), '
    f'sendo {zeros} pontuais e {contagens[0] - zeros} com atraso de 1 a {int(bordas[1]) - 1} min'
)

## 3. Tabelas comparativas de dez registros (seção A.1.6)

Os dez registros são fixados por `RESPONDENT_ID`, chave de integração das quatro fontes e única na
base, e nunca por posição. O motivo é concreto: `descobrir_fontes`, em
`scripts/preprocessamento_nps.py`, lista os arquivos com `Path.iterdir()`, que não garante ordem
alguma. Dependendo do sistema de arquivos, `NPS_01` a `NPS_04` são concatenados em ordens diferentes,
e um `df.head(10)` devolve dez registros completamente diferentes sem que nada acuse. Selecionar pela
chave torna as duas tabelas auditáveis linha a linha.

Os dez escolhidos são os de menor `RESPONDENT_ID` da base, sem seleção por valor, para que a amostra
não fique escolhida a favor do argumento.

In [ ]:
VARIAVEIS = ['TEMPO_VOO', 'ATRASO_CHEGADA', 'QTDE_VIAGENS_12M']
ESCALONADAS = ['TEMPO_VOO_ESC', 'ATRASO_CHEGADA_ESC', 'QTDE_VIAGENS_12M_ESC']

REGISTROS = [
    28211922, 28211924, 28211938, 28211953, 28211984,
    28211986, 28211989, 28211999, 28212003, 28212029,
]

encontrados = int(df['RESPONDENT_ID'].isin(REGISTROS).sum())
assert encontrados == len(REGISTROS), (
    f'esperava os {len(REGISTROS)} registros publicados na seção A.1.6 e encontrei {encontrados}'
)

amostra = df[df['RESPONDENT_ID'].isin(REGISTROS)].set_index('RESPONDENT_ID').loc[REGISTROS]

originais = amostra[VARIAVEIS]
escalonados = amostra[ESCALONADAS].round(4)

print('Dados originais')
display(originais)
print()
print('Dados escalonados')
display(escalonados)

Três verificações rápidas sobre a amostra, que o texto da A.1.6 comenta: quantos dos dez registros têm
atraso zero, quantos ficam acima da média de `TEMPO_VOO`, e onde a coluna normalizada se posiciona no
intervalo disponível.

In [ ]:
pontuais = int((amostra['ATRASO_CHEGADA'] == 0).sum())
print(f'registros da amostra com atraso zero: {pontuais}, todos em {escalonados["ATRASO_CHEGADA_ESC"].min():.4f}')

acima = amostra[amostra['TEMPO_VOO'] > MEDIA_TEMPO_VOO]
print(f'registros da amostra acima da média de TEMPO_VOO ({MEDIA_TEMPO_VOO}): {len(acima)}')
for respondent_id, linha in acima.iterrows():
    print(f'  {respondent_id}: {linha["TEMPO_VOO"]:.0f} min -> {linha["TEMPO_VOO_ESC"]:+.4f}')

print(
    f'QTDE_VIAGENS_12M escalonada na amostra: de {escalonados["QTDE_VIAGENS_12M_ESC"].min():.4f} '
    f'a {escalonados["QTDE_VIAGENS_12M_ESC"].max():.4f}'
)